### Silver Layer Initialization

This section prepares the Silver transformation by loading required PySpark modules, defining the Bronze/Silver table names, importing the Bronze dataset, and declaring a strict Silver schema to ensure type stability and consistent structure before applying any transformations.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    IntegerType, DateType, TimestampType
)
from pyspark.sql.window import Window

bronze_tbl = "sec_def14a_bronze"
silver_tbl = "sec_def14a_silver"

# Load Bronze table
bronze = spark.table(bronze_tbl)

# Define Silver schema explicitly for stability and type safety
silver_schema = StructType([
    StructField("ticker", StringType(), True),
    StructField("cik", StringType(), False),
    StructField("accession_number", StringType(), False),
    StructField("filing_date", DateType(), True),
    StructField("fiscal_year_end", DateType(), True),
    StructField("company_name", StringType(), True),
    StructField("form", StringType(), True),
    StructField("peo_name", StringType(), True),
    StructField("peo_total_comp", DoubleType(), True),
    StructField("peo_actually_paid_comp", DoubleType(), True),
    StructField("neo_avg_total_comp", DoubleType(), True),
    StructField("neo_avg_actually_paid_comp", DoubleType(), True),
    StructField("total_shareholder_return", DoubleType(), True),
    StructField("peer_group_tsr", DoubleType(), True),
    StructField("net_income", DoubleType(), True),
    StructField("company_selected_measure", StringType(), True),
    StructField("company_selected_measure_value", DoubleType(), True),
    StructField("fiscal_year", IntegerType(), True),
    StructField("ingestion_ts", TimestampType(), True),
    StructField("is_latest_peo", IntegerType(), True),
    StructField("is_latest_neo", IntegerType(), True),
    StructField("is_latest_net_income", IntegerType(), True)
])

# Start with Bronze
df = bronze

### Data Cleaning, Fiscal Year Derivation, Deduplication, and Latest‑Record Identification

This section standardizes numeric fields by converting zero values to NULL, derives a consistent fiscal year from the filing’s year‑end date, adds an ingestion timestamp when missing, removes duplicate records per CIK and fiscal year, and applies window functions to identify the most recent valid PEO, NEO, and Net Income entries for each company.

In [ ]:
# Replace 0 values with NULL for numeric fields
numeric_fields = [
    "peo_total_comp",
    "peo_actually_paid_comp",
    "neo_avg_total_comp",
    "neo_avg_actually_paid_comp",
    "total_shareholder_return",
    "peer_group_tsr",
    "net_income",
    "company_selected_measure_value"
]

for c in numeric_fields:
    if c in df.columns:
        df = df.withColumn(
            c,
            F.when(F.col(c) == 0, F.lit(None)).otherwise(F.col(c)).cast("double")
        )

# Compute fiscal year
if "fiscal_year_end" in df.columns:
    df = df.withColumn(
        "fiscal_year",
        F.when(F.col("fiscal_year_end").isNull(), F.lit(None).cast("int"))
         .when(F.month(F.col("fiscal_year_end")) >= 7,
               F.year(F.col("fiscal_year_end")))
         .otherwise(F.year(F.col("fiscal_year_end")) - 1)
    )
else:
    df = df.withColumn("fiscal_year_end", F.lit(None).cast("date"))
    df = df.withColumn("fiscal_year", F.lit(None).cast("int"))

# Add ingestion timestamp if missing
if "ingestion_ts" not in df.columns:
    df = df.withColumn("ingestion_ts", F.current_timestamp())

# Deduplication: one row per cik + fiscal_year
w = Window.partitionBy("cik", "fiscal_year").orderBy("ticker")
df = df.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

# Latest Flags (PEO, NEO, Net Income)


w_peo = Window.partitionBy("cik").orderBy(
    F.col("peo_total_comp").isNull().asc(),
    F.col("fiscal_year").desc()
)

w_neo = Window.partitionBy("cik").orderBy(
    F.col("neo_avg_total_comp").isNull().asc(),
    F.col("fiscal_year").desc()
)

w_net = Window.partitionBy("cik").orderBy(
    F.col("net_income").isNull().asc(),
    F.col("fiscal_year").desc()
)

df = (
    df
    .withColumn("rn_peo", F.row_number().over(w_peo))
    .withColumn("rn_neo", F.row_number().over(w_neo))
    .withColumn("rn_net", F.row_number().over(w_net))
    .withColumn("is_latest_peo", F.when(F.col("rn_peo") == 1, 1).otherwise(0))
    .withColumn("is_latest_neo", F.when(F.col("rn_neo") == 1, 1).otherwise(0))
    .withColumn("is_latest_net_income", F.when(F.col("rn_net") == 1, 1).otherwise(0))
)

# clean numeric columns to replace NaN with null
def clean_numeric_columns(df):
    numeric_types = ["double", "float", "integer", "bigint", "long", "decimal"]

    cleaned_cols = []
    for c, dtype in df.dtypes:
        if dtype in numeric_types:
            cleaned_cols.append(
                F.when(
                    F.isnan(F.col(c)) |
                    (F.col(c) == float("inf")) |
                    (F.col(c) == float("-inf")),
                    None
                ).otherwise(F.col(c)).alias(c)
            )
        else:
            cleaned_cols.append(F.col(c))

    return df.select(cleaned_cols)
    
df = clean_numeric_columns(df)

### Schema Normalization and Final Silver Table Write

This section guarantees that the DataFrame fully conforms to the predefined Silver schema by adding any missing columns, enforcing correct data types, and selecting fields in a stable order. Once the schema is aligned, the cleaned and standardized dataset is written to the Silver Delta table using an overwrite operation to ensure consistency across pipeline runs.

In [1]:
# Ensure all schema fields exist and cast correctly
silver_cols = [f.name for f in silver_schema.fields]

for field in silver_schema.fields:
    if field.name not in df.columns:
        df = df.withColumn(field.name, F.lit(None).cast(field.dataType))

df_silver = df.select([F.col(c).cast(silver_schema[c].dataType) for c in silver_cols])

# Write Silver table
df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_tbl)


StatementMeta(, f2835796-e3b3-49f0-abeb-5b093d469dad, 3, Finished, Available, Finished, False)

Silver table overwritten with latest flags.


SynapseWidget(Synapse.DataFrame, 93a7f77d-346c-4c05-899d-61c6f9f420e9)